# Data Exploration

## Project Introduction

This project investigates whether deep learning can transform sketch-like furniture edge maps into plausible RGB furniture images.

Each original furniture photograph is converted into a Canny edge representation. The edge map serves as the model input, while the corresponding photograph serves as the target image. This creates paired data suitable for supervised image-to-image generation.

The project extends the earlier [Furniture-Sketch-Classifier](https://github.com/Vanya2307/Furniture-Sketch-Classifier) study. That work used Canny edge detection, handcrafted HOG features, and classical machine learning models to classify five furniture categories. The present research shifts the task from sketch classification to sketch-to-image generation using deep learning. The earlier classifier provides a historical reference and will later be reused as an auxiliary semantic evaluator: generated RGB images will be converted into Canny edge maps and classified to test whether the intended furniture category is preserved.

Two generative models will be compared. A reconstruction-only U-Net trained with an $L_1$ objective will serve as the baseline. The Pix2Pix conditional GAN will use the same U-Net generator architecture together with a PatchGAN discriminator and will combine reconstruction and adversarial losses. This controlled comparison is intended to isolate the contribution of adversarial training to visual realism.

Data come from the Bonn Furniture Styles Dataset (Aggarwal et al., 2018). The initial scope includes the official `beds` and `dressers` categories because they are directly relevant to bedroom furniture design and provide a useful contrast in visual structure. Dressers tend to contain rectilinear forms and repeated drawer-like elements, whereas beds show greater variation in proportions, frames, and subtypes. Both categories contain multiple furniture subtypes and styles, which will be examined before the final cleaned dataset is defined.

**Dataset:** Bonn Furniture Styles Dataset, Aggarwal et al. (2018)  
**Original categories:** beds, chairs, dressers, lamps, sofas, tables  
**Initially selected categories:** beds, dressers  
**Input:** Canny edge map  
**Target:** corresponding RGB furniture photograph  
**Image size:** 256 × 256 pixels  
**Split:** predefined train / validation / test files  
**Planned comparison:** reconstruction-only U-Net vs. Pix2Pix using the same U-Net generator architecture

## Notebook Overview

Before generating paired images or training deep learning models, the dataset must be examined and validated carefully.

This notebook focuses on:

- loading and parsing the predefined dataset split files;
- selecting the `beds` and `dressers` categories;
- inspecting category, furniture-style, and product-subtype distributions;
- checking image paths, file integrity, and technical image properties;
- identifying duplicate paths, exact image duplicates, and possible near duplicates;
- verifying that duplicated content does not cause leakage between dataset splits;
- examining ambiguous product subtypes inside the official categories;
- defining a cleaned and reproducible dataset for paired image generation;
- saving portable metadata for the following notebooks.

## Setup and Imports

In [1]:
import hashlib
import random
import re
import sys
from pathlib import Path

import cv2
import imagehash
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm


PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_colwidth", 120)

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"OpenCV: {cv2.__version__}")

Python: 3.12.3
NumPy: 2.5.1
pandas: 3.0.3
OpenCV: 5.0.0


## Dataset Access and Split Loading

The analysis uses the original train, validation, and test definitions provided with the Bonn Furniture Styles Dataset. Raw images are not distributed with this repository. The notebook expects the extracted dataset to be available under `data/raw/bonn_furniture_styles/`.

In [5]:
from src.data_utils import (IMAGE_ROOT, SELECTED_CATEGORIES, load_dataset_splits,)


available_categories = sorted(path.name for path in IMAGE_ROOT.iterdir() if path.is_dir())

missing_categories = sorted(set(SELECTED_CATEGORIES) - set(available_categories))

if missing_categories:
    raise FileNotFoundError(f"Missing selected categories: {missing_categories}")

dataset_metadata = load_dataset_splits()

print(f"Categories found: {', '.join(available_categories)}")
print(f"Loaded records: {len(dataset_metadata):}")

display(
    dataset_metadata
    .groupby("split", sort=False)
    .size()
    .rename("images")
    .to_frame()
)

preview_columns = [
    "split",
    "relative_path",
    "category",
    "style",
    "product_subtype",
    "metadata_style",
    "metadata_field_count",
    "styles_match",

]

display(dataset_metadata[preview_columns].head())

Categories found: beds, chairs, dressers, lamps, sofas, tables
Loaded records: 90085


,images
split,
train,61273
validation,10800
test,18012


,split,relative_path,category,style,product_subtype,metadata_style,metadata_field_count,styles_match
0,train,houzz/dressers/Transitional/6609transitional-dressers.jpg,dressers,Transitional,Dressers,Transitional,6,True
1,train,houzz/dressers/Contemporary/625contemporary-dressers.jpg,dressers,Contemporary,Dressers,Contemporary,6,True
2,train,houzz/chairs/Midcentury/18157midcentury-armchairs-and-accent-chairs.jpg,chairs,Midcentury,Armchairs And Accent Chairs,Midcentury,6,True
3,train,houzz/lamps/Contemporary/1173contemporary-table-lamps.jpg,lamps,Contemporary,Table Lamps,Contemporary,6,True
4,train,houzz/lamps/Transitional/25709transitional-floor-lamps.jpg,lamps,Transitional,Floor Lamps,Transitional,6,True


In [6]:
#One final check of complete dataset

metadata_schema = (
    dataset_metadata["metadata_field_count"]
    .value_counts()
    .sort_index()
    .rename_axis("metadata_fields")
    .rename("records")
    .to_frame()
)
style_mismatches = (~dataset_metadata["styles_match"]).sum()

display(metadata_schema)
print(f"Style label mismatches: {style_mismatches:}")

,records
metadata_fields,
2,12254
6,77187
7,342
8,286
9,15
10,1


Style label mismatches: 0


In [7]:
#Records of two fields of metadata

two_field_records = dataset_metadata[
    dataset_metadata["metadata_field_count"] == 2
]

print(f"Records with two metadata fields: {len(two_field_records):}")

display(
    two_field_records[
        [
            "category",
            "product_subtype",
            "metadata_style",
            "metadata_attributes",
        ]
    ].head(10)
)

display(
    pd.crosstab(
        dataset_metadata["category"],
        dataset_metadata["metadata_field_count"],
    )
)

Records with two metadata fields: 12254


,category,product_subtype,metadata_style,metadata_attributes
5,tables,tables,Scandinavian,
18,beds,beds,Contemporary,
44,beds,beds,Transitional,
45,beds,beds,Contemporary,
49,beds,beds,Rustic,
54,beds,beds,Transitional,
73,lamps,lamps,Craftsman,
75,lamps,lamps,Craftsman,
88,beds,beds,Mediterranean,
101,beds,beds,Mediterranean,


metadata_field_count,2,6,7,8,9,10
category,,,,,,
beds,6578,0,0,0,0,0
chairs,261,21615,119,51,6,1
dressers,170,7678,13,11,0,0
lamps,4971,27104,135,185,7,0
sofas,19,4037,4,20,0,0
tables,255,16753,71,19,2,0


**Observation**

- All 90085 split records were loaded successfully: 61273 training, 10800 validation, and 18012 test records.
- All six expected furniture category directories were found.
- The metadata schema varies from two to ten fields. Most records contain six fields (77187; 85.7%), while 12254 records (13.6%) contain only two.
- Metadata availability is strongly category-dependent. Every bed record contains only two fields, and lamps account for another 4971 two-field records. In contrast, most dresser records contain six fields.
- The two-field records contain only product subtype and furniture style; optional product attributes are absent. This does not affect image generation because these attributes are not used as model inputs, but it limits metadata-based analysis, especially for beds.
- The official furniture category is therefore derived from the image directory path rather than the optional metadata.
- Style labels agree across the split files, directory paths, and metadata for all records.
- No records are removed during loading. Duplicate paths, duplicate images, and cross-split leakage will be investigated separately before the cleaned subset is created.